# First part setting up Expert System

## importing the libraries

In [ ]:
# installing the libraries (only need to be done once)
# delete # to run
# ! pip install ipywidgets
# ! pip install IPython
# ! pip install collections
# ! pip install heapq
# ! pip install time

In [ ]:
# Improting the Library used in the project (run each time to use)

from collections import deque
from collections import Counter
import heapq
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

## Building the knowledge base

In [ ]:
# First we need to build the Knowledge base for the search space here we called it Diagnosis
# We also build the goal set with all unique goals with our final diagnosis

Diagnosis = {'Fever': ['Cough', 'Diarrhea'] ,
             'Sneezing': ['Runny nose'] ,
             'Frequent urination': ['Burning urination', 'Increase thirst'] ,
             'Cough': ['Shortness of breath', 'Sore throat', 'Common Cold'] ,
             'Diarrhea': ['Vomiting', 'Gastroenteritis'],
             'Runny nose': ['Fatigue', 'The Flu'],
             'Runny nose': ['Cough', 'Itchy eyes'],
             'Burning urination': ['Lower abdominal pain', 'UTI'],
             'Increase thirst': ['Fatigue', 'Diabetes'],
             'Shortness of breath': ['Loss of smell', 'Chest pain'],
             'Sore throat':['Runny nose', 'The Flu','Common Cold'],
             'Vomiting': ['Abdominal cramps', 'Gastroenteritis'],
             'Itchy eyes': ['Nasal congestion', 'Allergic Rhinitis'],
             'Lower abdominal pain': ['Cloudy urine', 'UTI'],
             'Fatigue':['Blurred vision', 'Diabetes','The Flu'],
             'Loss of smell': ['Covid19'],
             'Chest pain': ['Pneumonia'],
             'Abdominal cramps':['Gastroenteritis'],
             'Nasal congestion': ['Allergic Rhinitis'],
             'Cloudy urine': ['UTI'],
             'Blurred vision':['Diabetes'],
             'Diabetes':[],
             'Covid19':[],
             'Pneumonia': [],
             'The Flu':[],
             'Gastroenteritis': [],
             'Common Cold':[],
             'Allergic Rhinitis':[],
             'UTI':[],
             'Diabetes':[]
             }


goal = ['Gastroenteritis', 'UTI', 'Diabetes', 'The Flu', 'Covid19', 'Pneumonia',
        'Common Cold', 'Allergic Rhinitis']


# after we list our symbtoms and tratment
symptom_options = ['Fever', 'Diarrhea', 'Vomiting', 'Sneezing', 'Frequent urination','Cough', 'Runny nose', 'Burning urination', 'Increase thirst', 'Shortness of breath',
                   'Sore throat', 'Itchy eyes', 'Lower abdominal pain', 'Fatigue', 'Loss of smell', 'Chest pain','Abdominal cramps',
                   'Nasal congestion', 'Cloudy urine', 'Blurred vision']


Treatment = {'Gastroenteritis': '1. Drink oral rehydration solutions frequently \n2. Rest and eat light foods \n3. Take anti-vomiting medicine like Ondansetron if needed \n4. Seek help if dehydration symptoms appear',
             'UTI': '1. Drink plenty of water \n2. Take prescribed antibiotics such as Nitrofurantoin \n3. Do not delay treatment to avoid complications',
             'Diabetes' : '1. Follow a healthy diet \n2. Exercise regularly \n3. Take medications such as Metformin as prescribed \n4. Monitor your blood sugar levels regularly',
             'The Flu' : '1. Drink plenty of fluids (water, soup, warm drinks) \n2. Take Paracetamol or Ibuprofen to reduce fever \n3. Keep yourself warm and rest well',
             'Covid19': '1. Isolate yourself to avoid spreading infection \n2. Rest and drink plenty of fluids \n3. Take Paracetamol for fever \n4. Consult a doctor about antivirals like Paxlovid \n5. Seek urgent care if you have difficulty breathing',
             'Pneumonia' : '1. Take prescribed antibiotics such as Amoxicillin \n2. Get plenty of rest \n3. Drink fluids regularly \n4. Use Ibuprofen for pain and fever \n5. Go to the hospital if symptoms become severe',
             'Common Cold' : '1. Rest and stay hydrated \n2. Take Paracetamol if needed \n3. Use decongestants like Pseudoephedrine \n4. Drink warm fluids',
             'Allergic Rhinitis' : '1. Take antihistamines like Loratadine or Cetirizine \n2. Avoid dust, pollen, or other allergens \n3. Use nasal sprays if necessar'}

## building the Breadth First Search

In [ ]:
class Diagnosis_BFS:
  def __init__(self, Diagnosis, goal, symptom_options, Treatment):
    self.Diagnosis = Diagnosis
    self.goal = goal
    self.symptom_options = symptom_options
    self.Treatment = Treatment

    # here we set up what the patient will see and intract with
    self.checkboxes = [widgets.Checkbox(description = s) for s in symptom_options]
    self.submit= widgets.Button(description = 'Submit symptoms', disabled = True)
    self.output_area = widgets.Output()
    for cb in self.checkboxes:
      cb.observe(self._update_button_state, names = 'value')
    self.submit.on_click(self._on_submit_clicked)

  # to make sure the submit button only works if 3 or more symbtoms are chosen:
  def _update_button_state(self, change):
    count = sum(cb.value for cb in self.checkboxes)
    if count >= 3:
      self.submit.disabled = False
      self.submit.button_style = 'success'
    else:
      self.submit.disabled = True
      self.submit.button_style = ''
  # to get the output
  def _on_submit_clicked(self, b):
    with self.output_area:
      clear_output() # to clear output from previous patient
      selected = [cb.description.strip() for cb in self.checkboxes if cb.value]
      result = self.final_diagnosis_symptoms(selected)
      if result:
        print('The final diagnosis is: ', result)
        for d in result:
              treatment = self.Treatment.get(d, 'No treatment Known in the system')
              print(f'Treatment for {d}: {treatment}')
      else:
        print('No diagnosis found, please visit your doctor')

    # for the BFS firt this is a function that finds the closest possiable diagnosis
  def all_reach_diagnoses(self, symptom):
    queue = deque([symptom])
    visited = set([symptom])
    found = set () # a set for the diagnosis found
    while queue:
      current= queue.popleft()
      if current in self.goal:
        found.add(current)
      for neighbor in self.Diagnosis.get(current, []):
        if neighbor not in visited:
          visited.add(neighbor)
          queue.append(neighbor)
    return found

    # after finding the possiable diagnosis for the symptoms we now find the
  def final_diagnosis_symptoms(self,sym):
    s = len(sym)
    counter=Counter()
    for symptom in sym:
      reachable = self.all_reach_diagnoses(symptom)
      for diagnose in reachable:
        counter[diagnose] +=1
    if not counter:
      return []
    best_diagnose = max(counter.values())
    return ([d for d, count in counter.items() if count == best_diagnose and count >= s])

      # here is what the paitint would see
  def display_ui(self):
    print('Please select your symptoms (make sure to chose at least 3 symptoms):')
    for cb in self.checkboxes:
      display(cb)
    display(self.submit)
    display(self.output_area)


In [ ]:
# test for all symptoms for BFS
symptoms_to_test =  ['Fever', 'Diarrhea', 'Vomiting', 'Sneezing', 'Frequent urination','Cough', 'Runny nose', 'Burning urination', 'Increase thirst', 'Shortness of breath',
                   'Sore throat', 'Itchy eyes', 'Lower abdominal pain', 'Fatigue', 'Loss of smell', 'Chest pain','Abdominal cramps',
                   'Nasal congestion', 'Cloudy urine', 'Blurred vision']
app = Diagnosis_BFS(Diagnosis, goal, symptom_options, Treatment)
for s in symptoms_to_test:
    found = app.all_reach_diagnoses(s)
    print(f"{s} can reach: {found}")

## Building A* Search

In [ ]:
class Diagnosis_A_s:
  def __init__(self, Diagnosis, goal, symptom_options, Treatment):
    self.Diagnosis = Diagnosis
    self.goal = goal
    self.symptom_options = symptom_options
    self.Treatment = Treatment

    # here we set up what the patient will see and intract with
    self.checkboxes = [widgets.Checkbox(description = s) for s in symptom_options]
    self.submit= widgets.Button(description = 'Submit symptoms', disabled = True)
    self.output_area = widgets.Output()
    for cb in self.checkboxes:
      cb.observe(self._update_button_state, names = 'value')
    self.submit.on_click(self._on_submit_clicked)

  # to make sure the submit button only works if 3 or more symbtoms are chosen:
  def _update_button_state(self, change):
    count = sum(cb.value for cb in self.checkboxes)
    if count >= 3:
      self.submit.disabled = False
      self.submit.button_style = 'success'
    else:
      self.submit.disabled = True
      self.submit.button_style = ''
  # to get the output
  def _on_submit_clicked(self, b):
    with self.output_area:
      clear_output() # to clear output from previous patient
      selected = [cb.description.strip() for cb in self.checkboxes if cb.value]
      result = self.A_diagnoses(selected)
      if result:
        print('The final diagnosis is: ', result)
        for d in result:
              treatment = self.Treatment.get(d, 'No treatment Known in the system')
              print(f'Treatment for {d}: {treatment}')
      else:
        print('No diagnosis found, please visit your doctor')

    # we will us a BFS to get all poissible node to reach to calculate the huristic function
  def bfs_reachable(self, symptom):
    visited = set([symptom])
    queue = deque([symptom])

    while queue:
        vertex = queue.popleft()

        for neighbor in self.Diagnosis[vertex]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return visited

   # A * search
  def A_diagnoses(self, sym):
  # here we want the reachable sysmptom to get to diagnosis
    reachable = {s: self.bfs_reachable(s) for s in sym}

    # here we will count how many symptoms listed that are not in reachable
    # those symptoms are not wanted as they will effect the huristic
    def heuristic(node):
      extra = 0
      for s in sym:
        if node not in reachable[s]:
          extra +=1
      return extra

    open_set =[]
    diagnosis_found = []
    visisted = set()

    for s in sym:
    #                         (heuristic(s), the goal, symptoms to reach goal)
      heapq.heappush(open_set, (heuristic(s), 0, s))

    while open_set:
      f,g,current = heapq.heappop(open_set)

      if current in visisted:
        continue
      visisted.add(current)
      # to make sure all symptoms reached the goal
      if  current in goal and heuristic(current) == 0:
        ## if yes add to list
        if current not in diagnosis_found:
          diagnosis_found.append(current)

      for neighbor in self.Diagnosis.get(current,[]):
        if neighbor not in visisted:
          g_new = g +1
          f_new = g_new + heuristic(neighbor)
          heapq.heappush(open_set, (f_new, g_new, neighbor))
    return diagnosis_found


      # here is what the paitint would see
  def display_ui(self):
    print('Please select your symptoms (make sure to chose at least 3 symptoms):')
    for cb in self.checkboxes:
      display(cb)
    display(self.submit)
    display(self.output_area)

In [ ]:
# test for all symptoms using A*
symptoms_to_test =  ['Fever', 'Diarrhea', 'Vomiting', 'Sneezing', 'Frequent urination','Cough', 'Runny nose', 'Burning urination', 'Increase thirst', 'Shortness of breath',
                   'Sore throat', 'Itchy eyes', 'Lower abdominal pain', 'Fatigue', 'Loss of smell', 'Chest pain','Abdominal cramps',
                   'Nasal congestion', 'Cloudy urine', 'Blurred vision']
app = Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment)
for s in symptoms_to_test:
    found = app.A_diagnoses([s])
    print(f"{s} can reach: {found}")

### Getting heuristic using the same method used in search

#### the following is only 3 examples the rest of the heuristic were calculated and documented in the report

In [ ]:
# to get the heuristic Gastroenteritis
symptoms_to_test =  ['Fever', 'Diarrhea', 'Vomiting','Abdominal cramps']

app= Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment)
reachable = {s: app.bfs_reachable(s) for s in symptoms_to_test}
def heuristic(node):
  extra = 0
  for s in symptoms_to_test:
    if node not in reachable[s]:
      extra +=1
  return extra


for node in app.Diagnosis.keys():
  print(f"Heuristic for {node}: {heuristic(node)}")


In [ ]:
# to get the heuristic Covid19
symptoms_to_test =  ['Fever','Cough','Shortness of breath','Loss of smell']

app= Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment)
reachable = {s: app.bfs_reachable(s) for s in symptoms_to_test}
def heuristic(node):
  extra = 0
  for s in symptoms_to_test:
    if node not in reachable[s]:
      extra +=1
  return extra


for node in app.Diagnosis.keys():
  print(f"Heuristic for {node}: {heuristic(node)}")

In [ ]:
# to get the heuristic Pneumonia
symptoms_to_test =  ['Fever','Cough','Shortness of breath','Chest pain']

app= Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment)
reachable = {s: app.bfs_reachable(s) for s in symptoms_to_test}
def heuristic(node):
  extra = 0
  for s in symptoms_to_test:
    if node not in reachable[s]:
      extra +=1
  return extra


for node in app.Diagnosis.keys():
  print(f"Heuristic for {node}: {heuristic(node)}")

In [ ]:
# to get the heuristic Diabetes
symptoms_to_test =  ['Frequent urination','Increase thirst','Fatigue','Blurred vision']

app= Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment)
reachable = {s: app.bfs_reachable(s) for s in symptoms_to_test}
def heuristic(node):
  extra = 0
  for s in symptoms_to_test:
    if node not in reachable[s]:
      extra +=1
  return extra


for node in app.Diagnosis.keys():
  print(f"Heuristic for {node}: {heuristic(node)}")

## Comparing BFS to A*

### run time

In [ ]:
# search will be tested 3 times with different amount of symptoms and we will take the average
symptoms_to_test_1 =  ['Fever', 'Diarrhea', 'Vomiting']
symptoms_to_test_2 = ['Frequent urination','Increase thirst', 'Fatigue', 'Blurred vision']
symptoms_to_test_3 = ['Cough', 'Shortness of breath', 'Itchy eyes','Chest pain']

In [ ]:
# test for time to run BFS 1
start = time.process_time()
app = Diagnosis_BFS(Diagnosis, goal, symptom_options, Treatment)
for s in symptoms_to_test_1:
    found = app.all_reach_diagnoses(s)
end = time.process_time()
t1 = end - start
print(t1)

In [ ]:
# test for time to run BFS 2
start = time.process_time()
app = Diagnosis_BFS(Diagnosis, goal, symptom_options, Treatment)
for s in symptoms_to_test_2:
    found = app.all_reach_diagnoses(s)
end = time.process_time()
t2 = end - start
print(t2)

In [ ]:
# test for time to run BFS 3
start = time.process_time()
app = Diagnosis_BFS(Diagnosis, goal, symptom_options, Treatment)
for s in symptoms_to_test_3:
    found = app.all_reach_diagnoses(s)
end = time.process_time()
t3 = end - start
print(t3)

In [ ]:
Avg_time_BFS = (t1+ t2+ t3) / 3
print(Avg_time_BFS)

In [ ]:
# test for time to run A* 1
start = time.process_time()
app = Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment)
for s in symptoms_to_test_1:
    found = app.A_diagnoses([s])
end = time.process_time()
a_t1 = end - start
print(a_t1)

In [ ]:
# test for time to run A* 2
start = time.process_time()
app = Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment)
for s in symptoms_to_test_2:
    found = app.A_diagnoses([s])
end = time.process_time()
a_t2 = end - start
print(t2)

In [ ]:
# test for time to run A* 3
start = time.process_time()
app = Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment)
for s in symptoms_to_test_3:
    found = app.A_diagnoses([s])
end = time.process_time()
a_t3 = end - start
print(a_t3)

In [ ]:
Avg_time_A = (a_t1+ a_t2+ a_t3) / 3
print(Avg_time_A)

# Implementing the Expert System

## Implementing Breadth First Search

In [ ]:

Diagnosis_BFS(Diagnosis, goal, symptom_options, Treatment).display_ui()

## Implementing A*

In [ ]:
Diagnosis_A_s(Diagnosis, goal, symptom_options, Treatment).display_ui()